#### Pilot Test

In [3]:
!pip install google-play-scraper pandas tqdm pytz


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from __future__ import annotations

import hashlib
import logging
import re
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd
import pytz
from google_play_scraper import Sort, reviews
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Konfigurasi
# ---------------------------------------------------------------------------

WIB = pytz.timezone("Asia/Jakarta")


@dataclass(frozen=True)
class AppTarget:
    app_id: str
    app_name: str
    launch_date: datetime   # WIB-localized, inclusive (awal window)
    window_end: datetime    # WIB-localized, inclusive (akhir window)


TARGETS = [
    AppTarget(
        app_id="id.bni.wondr",
        app_name="wondr by BNI",
        launch_date=WIB.localize(datetime(2024, 7, 5, 0, 0, 0)),
        window_end=WIB.localize(datetime(2025, 7, 4, 23, 59, 59)),
    ),
    AppTarget(
        app_id="co.id.bankbsi.superapp",
        app_name="BYOND by BSI",
        launch_date=WIB.localize(datetime(2024, 11, 9, 0, 0, 0)),
        window_end=WIB.localize(datetime(2025, 11, 8, 23, 59, 59)),
    ),
]

OUTPUT_DIR = Path("./data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 200               # Maksimum per panggilan google-play-scraper
MAX_BATCHES_PER_APP = 10     # Safety ceiling: 1000 * 200 = 200k review
SLEEP_BETWEEN_BATCHES = 1.5    # Jeda antar batch (detik) agar tidak di-rate-limit

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------------

def hash_user(name: str) -> str:
    """Hash nama user untuk privacy — tetap bisa melacak user yang sama
    tanpa menyimpan PII mentah."""
    if not name:
        return ""
    return hashlib.sha256(name.encode("utf-8")).hexdigest()[:16]


def count_tokens(text: str) -> int:
    """Hitung jumlah kata secara kasar (whitespace-split setelah strip tanda baca)."""
    if not text:
        return 0
    stripped = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return len(stripped.split())


def to_wib(dt: datetime) -> datetime:
    """Konversi datetime (naive UTC dari API atau yang sudah aware) ke WIB."""
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    return dt.astimezone(WIB)


# ---------------------------------------------------------------------------
# Logika scraping
# ---------------------------------------------------------------------------

def scrape_app(target: AppTarget) -> tuple[pd.DataFrame, dict]:
    """
    Paginasi review (sort by NEWEST) sampai melewati tanggal launch.
    Return: (dataframe review dalam window, metadata scraping).
    """
    log.info("=" * 70)
    log.info("Scraping %s (%s)", target.app_name, target.app_id)
    log.info("Window: %s s/d %s (WIB)",
             target.launch_date.date(), target.window_end.date())
    log.info("=" * 70)

    all_rows = []
    continuation_token = None
    batches_fetched = 0
    total_raw_fetched = 0
    oldest_seen_wib = None
    newest_seen_wib = None
    stop_reason = "unknown"

    pbar = tqdm(total=MAX_BATCHES_PER_APP, desc=target.app_name, unit="batch")

    while batches_fetched < MAX_BATCHES_PER_APP:
        try:
            result, continuation_token = reviews(
                target.app_id,
                lang="id",
                country="id",
                sort=Sort.NEWEST,
                count=BATCH_SIZE,
                continuation_token=continuation_token,
            )
        except Exception as e:
            log.error("Batch %d gagal: %s. Sleep 10 detik lalu retry.",
                      batches_fetched, e)
            time.sleep(10)
            try:
                result, continuation_token = reviews(
                    target.app_id,
                    lang="id",
                    country="id",
                    sort=Sort.NEWEST,
                    count=BATCH_SIZE,
                    continuation_token=continuation_token,
                )
            except Exception as e2:
                log.error("Retry juga gagal: %s. Berhenti untuk app ini.", e2)
                stop_reason = f"fetch_error: {e2}"
                break

        batches_fetched += 1
        pbar.update(1)

        if not result:
            stop_reason = "empty_batch"
            break

        total_raw_fetched += len(result)

        batch_times_wib = [to_wib(r["at"]) for r in result]
        batch_oldest = min(batch_times_wib)
        batch_newest = max(batch_times_wib)

        if oldest_seen_wib is None or batch_oldest < oldest_seen_wib:
            oldest_seen_wib = batch_oldest
        if newest_seen_wib is None or batch_newest > newest_seen_wib:
            newest_seen_wib = batch_newest

        for r in result:
            review_dt_wib = to_wib(r["at"])

            # Hanya simpan review dalam window koleksi
            if not (target.launch_date <= review_dt_wib <= target.window_end):
                continue

            text = r.get("content") or ""

            # Handle replyContent dan repliedAt yang bisa None
            reply_date = r.get("repliedAt")
            if reply_date is not None:
                if reply_date.tzinfo is None:
                    reply_date = pytz.utc.localize(reply_date)
                reply_date_iso = reply_date.isoformat()
            else:
                reply_date_iso = None

            # Normalisasi date_utc (API biasanya return naive UTC)
            raw_dt = r["at"]
            if raw_dt.tzinfo is None:
                raw_dt = pytz.utc.localize(raw_dt)

            all_rows.append({
                "review_id": r.get("reviewId", ""),
                "app_id": target.app_id,
                "app_name": target.app_name,
                "user_name_hash": hash_user(r.get("userName", "")),
                "rating": r["score"],
                "review_text": text,
                "thumbs_up_count": r.get("thumbsUpCount", 0),
                "review_created_version": r.get("reviewCreatedVersion"),
                "reply_content": r.get("replyContent"),
                "reply_date": reply_date_iso,
                "date_utc": raw_dt.isoformat(),
                "date_wib": review_dt_wib.isoformat(),
                "date_local": review_dt_wib.date().isoformat(),
                "word_count": count_tokens(text),
                "char_count": len(text),
                "scrape_timestamp": datetime.now(WIB).isoformat(),
            })

        # Berhenti kalau sudah melewati tanggal launch
        if batch_oldest < target.launch_date:
            stop_reason = "passed_launch_date"
            break

        if continuation_token is None:
            stop_reason = "no_continuation_token"
            break

        time.sleep(SLEEP_BETWEEN_BATCHES)

    pbar.close()

    metadata = {
        "app_name": target.app_name,
        "app_id": target.app_id,
        "stop_reason": stop_reason,
        "batches_fetched": batches_fetched,
        "total_raw_fetched": total_raw_fetched,
        "total_in_window": len(all_rows),
        "oldest_review_seen_wib": oldest_seen_wib.isoformat() if oldest_seen_wib else None,
        "newest_review_seen_wib": newest_seen_wib.isoformat() if newest_seen_wib else None,
        "launch_date_wib": target.launch_date.isoformat(),
        "window_end_wib": target.window_end.isoformat(),
        "reached_launch_date": (
            oldest_seen_wib is not None and oldest_seen_wib <= target.launch_date
        ),
    }

    log.info("Alasan berhenti: %s", stop_reason)
    log.info("Total batch diambil: %d", batches_fetched)
    log.info("Total review mentah (semua tanggal): %d", total_raw_fetched)
    log.info("Total review dalam window: %d", len(all_rows))
    if oldest_seen_wib:
        log.info("Review terlama yang terlihat: %s", oldest_seen_wib.date())
    if not metadata["reached_launch_date"]:
        log.warning(
            "API BERHENTI SEBELUM MENCAPAI TANGGAL LAUNCH. "
            "Review terlama %s, target launch %s. "
            "Data mungkin tidak lengkap untuk periode awal.",
            oldest_seen_wib.date() if oldest_seen_wib else "N/A",
            target.launch_date.date()
        )

    return pd.DataFrame(all_rows), metadata


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    all_metadata = []

    for target in TARGETS:
        df, metadata = scrape_app(target)
        all_metadata.append(metadata)

        if df.empty:
            log.warning("Tidak ada review terkumpul untuk %s — cek app_id.",
                        target.app_name)
            continue

        safe_name = target.app_name.replace(" ", "_").replace("/", "-")
        output_path = OUTPUT_DIR / f"{safe_name}_raw.csv"
        df.to_csv(output_path, index=False, encoding="utf-8")
        log.info("Tersimpan: %s (%d baris)", output_path, len(df))

        # Ringkasan cepat untuk verifikasi
        log.info("  Distribusi rating: %s",
                 df["rating"].value_counts().sort_index().to_dict())
        log.info("  Rentang tanggal review: %s s/d %s",
                 df["date_local"].min(), df["date_local"].max())
        log.info("  Review dengan <5 kata: %d",
                 int((df["word_count"] < 5).sum()))
        log.info("  Review dengan <10 kata: %d",
                 int((df["word_count"] < 10).sum()))

    # Simpan metadata scraping — berguna untuk dokumentasi metodologi
    metadata_df = pd.DataFrame(all_metadata)
    metadata_path = OUTPUT_DIR / "scrape_metadata.csv"
    metadata_df.to_csv(metadata_path, index=False, encoding="utf-8")
    log.info("Metadata scraping tersimpan: %s", metadata_path)

    log.info("Selesai.")


if __name__ == "__main__":
    main()


2026-04-20 10:38:46,069 | INFO | ======================================================================
2026-04-20 10:38:46,071 | INFO | Scraping wondr by BNI (id.bni.wondr)
2026-04-20 10:38:46,073 | INFO | Window: 2024-07-05 s/d 2025-07-04 (WIB)
2026-04-20 10:38:46,085 | INFO | ======================================================================
wondr by BNI: 100%|█████████████████████████████████████████████████████████████████| 10/10 [04:33<00:00, 27.32s/batch]
2026-04-20 10:43:19,334 | INFO | Alasan berhenti: unknown
2026-04-20 10:43:19,361 | INFO | Total batch diambil: 10
2026-04-20 10:43:19,367 | INFO | Total review mentah (semua tanggal): 2000
2026-04-20 10:43:19,370 | INFO | Total review dalam window: 0
2026-04-20 10:43:19,372 | INFO | Review terlama yang terlihat: 2026-04-08
2026-04-20 10:43:19,380 | WARNING | API BERHENTI SEBELUM MENCAPAI TANGGAL LAUNCH. Review terlama 2026-04-08, target launch 2024-07-05. Data mungkin tidak lengkap untuk periode awal.
2026-04-20 10:43:19,5

#### Scraping

In [6]:
from __future__ import annotations

import hashlib
import logging
import re
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd
import pytz
from google_play_scraper import Sort, reviews
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Konfigurasi
# ---------------------------------------------------------------------------

WIB = pytz.timezone("Asia/Jakarta")


@dataclass(frozen=True)
class AppTarget:
    app_id: str
    app_name: str
    launch_date: datetime   # WIB-localized, inclusive (awal window)
    window_end: datetime    # WIB-localized, inclusive (akhir window)


TARGETS = [
    AppTarget(
        app_id="id.bni.wondr",
        app_name="wondr by BNI",
        launch_date=WIB.localize(datetime(2024, 7, 5, 0, 0, 0)),
        window_end=WIB.localize(datetime(2025, 7, 4, 23, 59, 59)),
    ),
    AppTarget(
        app_id="co.id.bankbsi.superapp",
        app_name="BYOND by BSI",
        launch_date=WIB.localize(datetime(2024, 11, 9, 0, 0, 0)),
        window_end=WIB.localize(datetime(2025, 11, 8, 23, 59, 59)),
    ),
]

OUTPUT_DIR = Path("./data/raw")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 200               # Maksimum per panggilan google-play-scraper
MAX_BATCHES_PER_APP = 1000     # Safety ceiling: 1000 * 200 = 200k review
SLEEP_BETWEEN_BATCHES = 1.5    # Jeda antar batch (detik) agar tidak di-rate-limit

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------------

def hash_user(name: str) -> str:
    """Hash nama user untuk privacy — tetap bisa melacak user yang sama
    tanpa menyimpan PII mentah."""
    if not name:
        return ""
    return hashlib.sha256(name.encode("utf-8")).hexdigest()[:16]


def count_tokens(text: str) -> int:
    """Hitung jumlah kata secara kasar (whitespace-split setelah strip tanda baca)."""
    if not text:
        return 0
    stripped = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return len(stripped.split())


def to_wib(dt: datetime) -> datetime:
    """Konversi datetime (naive UTC dari API atau yang sudah aware) ke WIB."""
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    return dt.astimezone(WIB)


# ---------------------------------------------------------------------------
# Logika scraping
# ---------------------------------------------------------------------------

def scrape_app(target: AppTarget) -> tuple[pd.DataFrame, dict]:
    """
    Paginasi review (sort by NEWEST) sampai melewati tanggal launch.
    Return: (dataframe review dalam window, metadata scraping).
    """
    log.info("=" * 70)
    log.info("Scraping %s (%s)", target.app_name, target.app_id)
    log.info("Window: %s s/d %s (WIB)",
             target.launch_date.date(), target.window_end.date())
    log.info("=" * 70)

    all_rows = []
    continuation_token = None
    batches_fetched = 0
    total_raw_fetched = 0
    oldest_seen_wib = None
    newest_seen_wib = None
    stop_reason = "unknown"

    pbar = tqdm(total=MAX_BATCHES_PER_APP, desc=target.app_name, unit="batch")

    while batches_fetched < MAX_BATCHES_PER_APP:
        try:
            result, continuation_token = reviews(
                target.app_id,
                lang="id",
                country="id",
                sort=Sort.NEWEST,
                count=BATCH_SIZE,
                continuation_token=continuation_token,
            )
        except Exception as e:
            log.error("Batch %d gagal: %s. Sleep 10 detik lalu retry.",
                      batches_fetched, e)
            time.sleep(10)
            try:
                result, continuation_token = reviews(
                    target.app_id,
                    lang="id",
                    country="id",
                    sort=Sort.NEWEST,
                    count=BATCH_SIZE,
                    continuation_token=continuation_token,
                )
            except Exception as e2:
                log.error("Retry juga gagal: %s. Berhenti untuk app ini.", e2)
                stop_reason = f"fetch_error: {e2}"
                break

        batches_fetched += 1
        pbar.update(1)

        if not result:
            stop_reason = "empty_batch"
            break

        total_raw_fetched += len(result)

        batch_times_wib = [to_wib(r["at"]) for r in result]
        batch_oldest = min(batch_times_wib)
        batch_newest = max(batch_times_wib)

        if oldest_seen_wib is None or batch_oldest < oldest_seen_wib:
            oldest_seen_wib = batch_oldest
        if newest_seen_wib is None or batch_newest > newest_seen_wib:
            newest_seen_wib = batch_newest

        for r in result:
            review_dt_wib = to_wib(r["at"])

            # Hanya simpan review dalam window koleksi
            if not (target.launch_date <= review_dt_wib <= target.window_end):
                continue

            text = r.get("content") or ""

            # Handle replyContent dan repliedAt yang bisa None
            reply_date = r.get("repliedAt")
            if reply_date is not None:
                if reply_date.tzinfo is None:
                    reply_date = pytz.utc.localize(reply_date)
                reply_date_iso = reply_date.isoformat()
            else:
                reply_date_iso = None

            # Normalisasi date_utc (API biasanya return naive UTC)
            raw_dt = r["at"]
            if raw_dt.tzinfo is None:
                raw_dt = pytz.utc.localize(raw_dt)

            all_rows.append({
                "review_id": r.get("reviewId", ""),
                "app_id": target.app_id,
                "app_name": target.app_name,
                "user_name_hash": hash_user(r.get("userName", "")),
                "rating": r["score"],
                "review_text": text,
                "thumbs_up_count": r.get("thumbsUpCount", 0),
                "review_created_version": r.get("reviewCreatedVersion"),
                "reply_content": r.get("replyContent"),
                "reply_date": reply_date_iso,
                "date_utc": raw_dt.isoformat(),
                "date_wib": review_dt_wib.isoformat(),
                "date_local": review_dt_wib.date().isoformat(),
                "word_count": count_tokens(text),
                "char_count": len(text),
                "scrape_timestamp": datetime.now(WIB).isoformat(),
            })

        # Berhenti kalau sudah melewati tanggal launch
        if batch_oldest < target.launch_date:
            stop_reason = "passed_launch_date"
            break

        if continuation_token is None:
            stop_reason = "no_continuation_token"
            break

        time.sleep(SLEEP_BETWEEN_BATCHES)

    pbar.close()

    metadata = {
        "app_name": target.app_name,
        "app_id": target.app_id,
        "stop_reason": stop_reason,
        "batches_fetched": batches_fetched,
        "total_raw_fetched": total_raw_fetched,
        "total_in_window": len(all_rows),
        "oldest_review_seen_wib": oldest_seen_wib.isoformat() if oldest_seen_wib else None,
        "newest_review_seen_wib": newest_seen_wib.isoformat() if newest_seen_wib else None,
        "launch_date_wib": target.launch_date.isoformat(),
        "window_end_wib": target.window_end.isoformat(),
        "reached_launch_date": (
            oldest_seen_wib is not None and oldest_seen_wib <= target.launch_date
        ),
    }

    log.info("Alasan berhenti: %s", stop_reason)
    log.info("Total batch diambil: %d", batches_fetched)
    log.info("Total review mentah (semua tanggal): %d", total_raw_fetched)
    log.info("Total review dalam window: %d", len(all_rows))
    if oldest_seen_wib:
        log.info("Review terlama yang terlihat: %s", oldest_seen_wib.date())
    if not metadata["reached_launch_date"]:
        log.warning(
            "API BERHENTI SEBELUM MENCAPAI TANGGAL LAUNCH. "
            "Review terlama %s, target launch %s. "
            "Data mungkin tidak lengkap untuk periode awal.",
            oldest_seen_wib.date() if oldest_seen_wib else "N/A",
            target.launch_date.date()
        )

    return pd.DataFrame(all_rows), metadata


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    all_metadata = []

    for target in TARGETS:
        df, metadata = scrape_app(target)
        all_metadata.append(metadata)

        if df.empty:
            log.warning("Tidak ada review terkumpul untuk %s — cek app_id.",
                        target.app_name)
            continue

        safe_name = target.app_name.replace(" ", "_").replace("/", "-")
        output_path = OUTPUT_DIR / f"{safe_name}_raw.csv"
        df.to_csv(output_path, index=False, encoding="utf-8")
        log.info("Tersimpan: %s (%d baris)", output_path, len(df))

        # Ringkasan cepat untuk verifikasi
        log.info("  Distribusi rating: %s",
                 df["rating"].value_counts().sort_index().to_dict())
        log.info("  Rentang tanggal review: %s s/d %s",
                 df["date_local"].min(), df["date_local"].max())
        log.info("  Review dengan <5 kata: %d",
                 int((df["word_count"] < 5).sum()))
        log.info("  Review dengan <10 kata: %d",
                 int((df["word_count"] < 10).sum()))

    # Simpan metadata scraping — berguna untuk dokumentasi metodologi
    metadata_df = pd.DataFrame(all_metadata)
    metadata_path = OUTPUT_DIR / "scrape_metadata.csv"
    metadata_df.to_csv(metadata_path, index=False, encoding="utf-8")
    log.info("Metadata scraping tersimpan: %s", metadata_path)

    log.info("Selesai.")


if __name__ == "__main__":
    main()


2026-04-20 10:49:44,845 | INFO | ======================================================================
2026-04-20 10:49:44,846 | INFO | Scraping wondr by BNI (id.bni.wondr)
2026-04-20 10:49:44,849 | INFO | Window: 2024-07-05 s/d 2025-07-04 (WIB)
2026-04-20 10:49:44,849 | INFO | ======================================================================
wondr by BNI:  46%|████████████████████████████▍                                 | 459/1000 [21:22<25:11,  2.79s/batch]
2026-04-20 11:11:06,868 | INFO | Alasan berhenti: passed_launch_date
2026-04-20 11:11:06,870 | INFO | Total batch diambil: 459
2026-04-20 11:11:06,871 | INFO | Total review mentah (semua tanggal): 91800
2026-04-20 11:11:06,872 | INFO | Total review dalam window: 44627
2026-04-20 11:11:06,874 | INFO | Review terlama yang terlihat: 2024-07-04
2026-04-20 11:11:08,266 | INFO | Tersimpan: data\raw\wondr_by_BNI_raw.csv (44627 baris)
2026-04-20 11:11:08,293 | INFO |   Distribusi rating: {1: 8313, 2: 2006, 3: 2198, 4: 2492, 5: 2961

In [7]:
import pandas as pd

wondr = pd.read_csv('data/raw/wondr_by_BNI_raw.csv')
byond = pd.read_csv('data/raw/BYOND_by_BSI_raw.csv')

# 1. Short reviews di subset rating 1-2 saja
for name, df in [('wondr', wondr), ('BYOND', byond)]:
    neg = df[df['rating'].isin([1, 2])]
    print(f"{name} negatif: {len(neg)}")
    print(f"  <5 kata: {(neg['word_count'] < 5).sum()} ({(neg['word_count'] < 5).mean()*100:.1f}%)")
    print(f"  <10 kata: {(neg['word_count'] < 10).sum()} ({(neg['word_count'] < 10).mean()*100:.1f}%)")
    print()

# 2. Distribusi temporal review negatif per bulan
for name, df in [('wondr', wondr), ('BYOND', byond)]:
    neg = df[df['rating'].isin([1, 2])].copy()
    neg['month'] = pd.to_datetime(neg['date_local']).dt.to_period('M')
    print(f"{name} negatif per bulan:")
    print(neg['month'].value_counts().sort_index())
    print()

# 3. Distribusi versi app — useful untuk korelasi dengan release
for name, df in [('wondr', wondr), ('BYOND', byond)]:
    print(f"{name} top 10 versi:")
    print(df['review_created_version'].value_counts().head(10))
    print()

# 4. Sample review untuk inspeksi cepat
print("Sample wondr negatif:")
print(wondr[wondr['rating'].isin([1,2])].sample(5)[['rating', 'review_text']])
print("\nSample BYOND negatif:")
print(byond[byond['rating'].isin([1,2])].sample(5)[['rating', 'review_text']])

wondr negatif: 10319
  <5 kata: 1338 (13.0%)
  <10 kata: 3669 (35.6%)

BYOND negatif: 22935
  <5 kata: 3524 (15.4%)
  <10 kata: 8524 (37.2%)

wondr negatif per bulan:
month
2024-07     275
2024-08     378
2024-09     539
2024-10     796
2024-11    2969
2024-12    1519
2025-01     819
2025-02     736
2025-03     613
2025-04     403
2025-05     460
2025-06     653
2025-07     159
Freq: M, Name: count, dtype: int64

BYOND negatif per bulan:
month
2024-11     233
2024-12    1846
2025-01    2029
2025-02    8055
2025-03    1293
2025-04     624
2025-05     621
2025-06    1602
2025-07     931
2025-08    1604
2025-09    3057
2025-10     841
2025-11     199
Freq: M, Name: count, dtype: int64

wondr top 10 versi:
review_created_version
1.3.1    9262
1.3.0    7852
1.2.0    6682
1.0.2    5802
1.3.2    5174
1.0.1    2797
1.0.3    2373
1.4.0     167
0.1.4      15
Name: count, dtype: int64

BYOND top 10 versi:
review_created_version
1.0.4    14693
1.1.2     6942
1.0.5     6631
1.0.3     5754
1.1.0    